In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/2026-10-22.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      126 non-null    int64  
 1   name    126 non-null    str    
 2   price   126 non-null    float64
 3   date    126 non-null    str    
dtypes: float64(1), int64(1), str(2)
memory usage: 4.1 KB


In [2]:
df.head()


,id,name,price,date
0,4753925,MSI VGA GeForce RTX 3050 VENTUS 2X E OC 6 GB,269.0,2026-10-22
1,4796691,PNY VGA Nvidia RTX PRO 6000 Blackwell 96 GB,19999.0,2026-10-22
2,5339936,Asus VGA GeForce RTX 5090 ROG Astral Edition 2...,7999.0,2026-10-22
3,4791908,MSI VGA GeForce RTX RTX 5060 SHADOW 2X OC 8 GB,429.0,2026-10-22
4,4810066,Sapphire VGA Radeon RX 9060 XT NITRO+ OC 16 GB,759.0,2026-10-22


In [3]:
df["name"] = df["name"].str.replace("VGA ", "")
df["brand"] = df["name"].str.split().str[0]


In [4]:
print(df["brand"])

0           MSI
1           PNY
2          Asus
3           MSI
4      Sapphire
         ...   
121    Sapphire
122        Asus
123        Asus
124        Asus
125        Asus
Name: brand, Length: 126, dtype: object


In [5]:
df["brand"].value_counts()

brand
Gigabyte    38
Asus        24
MSI         19
PNY         19
Sapphire    13
Asrock       6
AMD          5
Intel        1
XFX          1
Name: count, dtype: int64

In [6]:
df[df["brand"] == "VGA"]

,id,name,price,date,brand


In [7]:
df["name"].str.contains("Radeon|AMD", case = False).sum()

np.int64(35)

In [8]:
df["name"].str.contains("GeForce|Nvidia", case = False).sum()

np.int64(83)

In [9]:
df["name"].str.contains("Intel|Arc", case = False).sum()

np.int64(3)

In [10]:
amd = df["name"].str.contains("Radeon|AMD", case = False)
nvidia = df["name"].str.contains("GeForce|Nvidia", case = False)
intel = df["name"].str.contains("Intel|Arc", case = False)
df[~(amd|nvidia|intel)]

,id,name,price,date,brand
64,4590023,PNY Quadro A400 Low Profile 4 GB,349.0,2026-10-22,PNY
99,4590031,PNY Quadro A1000 Low Profile 8 GB,799.0,2026-10-22,PNY
115,4042328,PNY Quadro T400 Low Profile 4 GB,199.0,2026-10-22,PNY
117,3494659,PNY Quadro P620 V2 2 GB,199.9,2026-10-22,PNY
118,2902818,PNY Quadro P620 2GB GDDR5,249.9,2026-10-22,PNY


In [11]:
df["name"].str.contains("GeForce|Nvidia|PNY", case = False).sum()
nvidia = df["name"].str.contains("GeForce|Nvidia|PNY", case = False)

In [12]:
df[~(amd|nvidia|intel)]

,id,name,price,date,brand


In [13]:
amd.sum() + nvidia.sum() + intel.sum()

np.int64(126)

In [14]:
df.loc[amd, "gpu_maker"] = "AMD"
df.loc[nvidia, "gpu_maker"] = "NVIDIA"
df.loc[intel, "gpu_maker"] = "INTEL"

In [15]:
df["gpu_maker"].value_counts()

gpu_maker
NVIDIA    88
AMD       35
INTEL      3
Name: count, dtype: int64

In [16]:
df.head(10)

,id,name,price,date,brand,gpu_maker
0,4753925,MSI GeForce RTX 3050 VENTUS 2X E OC 6 GB,269.0,2026-10-22,MSI,NVIDIA
1,4796691,PNY Nvidia RTX PRO 6000 Blackwell 96 GB,19999.0,2026-10-22,PNY,NVIDIA
2,5339936,Asus GeForce RTX 5090 ROG Astral Edition 20 32 GB,7999.0,2026-10-22,Asus,NVIDIA
3,4791908,MSI GeForce RTX RTX 5060 SHADOW 2X OC 8 GB,429.0,2026-10-22,MSI,NVIDIA
4,4810066,Sapphire Radeon RX 9060 XT NITRO+ OC 16 GB,759.0,2026-10-22,Sapphire,AMD
5,4810090,Sapphire Radeon RX 9060 XT PULSE OC 16 GB,719.0,2026-10-22,Sapphire,AMD
6,4699440,Sapphire Radeon RX 9070 XT NITRO+ OC 16 GB,1199.0,2026-10-22,Sapphire,AMD
7,4820118,Gigabyte GeForce GT 710D3 2 GB,69.9,2026-10-22,Gigabyte,NVIDIA
8,5337283,Asus GeForce RTX 5060 Dual OC Advanced 8 GB,599.0,2026-10-22,Asus,NVIDIA
9,5164052,Asus Radeon RX 9070 GRE PRIME OC 12 GB,749.0,2026-10-22,Asus,AMD


In [17]:
amd_pro_cards= df["name"].str.contains("Radeon pro|Radeon AI PRO", case = False)
nvidia_pro_cards = df["name"].str.contains("Quadro|RTX PRO", case = False)
intel_pro_cards = df["name"].str.contains("Arc Pro", case = False)


In [18]:
df["is_pro"] = (amd_pro_cards|nvidia_pro_cards|intel_pro_cards)

## Separating professional and gaming cards

**Separation is based on the manufacturers' professional product lines: Quadro and RTX PRO (NVIDIA), Radeon Pro (AMD) and Arc Pro (Intel). Use does not matter: a gaming card such as the RTX 5080 still counts as gaming, even if it is used for professional work.**

In [19]:
df["is_pro"].value_counts()

is_pro
False    101
True      25
Name: count, dtype: int64

In [20]:
df["vram"] = df["name"].str.extract(r"(\d+)\s?GB?\b").astype(float)

In [21]:
df["vram"].value_counts()

vram
16.0    39
8.0     31
12.0    17
32.0    13
6.0      4
2.0      4
4.0      4
48.0     4
24.0     3
96.0     2
20.0     2
10.0     1
Name: count, dtype: int64

In [22]:
df[df["vram"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram
12,4761812,MSI GeForce RTX 5080 Ventus 3X White OC,1799.0,2026-10-22,MSI,NVIDIA,False,NaN
103,4500148,PNY NVIDIA RTX PRO SYNC,999.0,2026-10-22,PNY,NVIDIA,True,NaN


In [23]:
is_5080 = df["name"].str.contains("5080")

In [24]:
df[is_5080]

,id,name,price,date,brand,gpu_maker,is_pro,vram
12,4761812,MSI GeForce RTX 5080 Ventus 3X White OC,1799.0,2026-10-22,MSI,NVIDIA,False,NaN
21,4706048,Gigabyte GeForce RTX 5080 AORUS MASTER 16 GB,2099.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
24,4688619,Asus GeForce RTX 5080 TUF GAMING OC 16 GB,1999.0,2026-10-22,Asus,NVIDIA,False,16.0
51,4706056,Gigabyte GeForce RTX 5080 GAMING OC 16 GB,1999.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
52,4706064,Gigabyte GeForce RTX 5080 AERO OC SFF 16 GB,2299.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
54,5340772,Gigabyte GeForce RTX 5080 AORUS INFINITY 16 GB,2499.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
55,4706072,Gigabyte GeForce RTX 5080 WINDFORCE OC SFF 16 GB,1949.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
125,4866029,Asus GeForce RTX 5080 ROG ASTRAL GAMING OC WHI...,2249.0,2026-10-22,Asus,NVIDIA,False,16.0


In [25]:
df[df["vram"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram
12,4761812,MSI GeForce RTX 5080 Ventus 3X White OC,1799.0,2026-10-22,MSI,NVIDIA,False,NaN
103,4500148,PNY NVIDIA RTX PRO SYNC,999.0,2026-10-22,PNY,NVIDIA,True,NaN


In [26]:
df["vram"].value_counts()

vram
16.0    39
8.0     31
12.0    17
32.0    13
6.0      4
2.0      4
4.0      4
48.0     4
24.0     3
96.0     2
20.0     2
10.0     1
Name: count, dtype: int64

## Filling in missing VRAM

Two rows have no VRAM after the extraction:

- **MSI GeForce RTX 5080 Ventus 3X White OC** (id 4761812): the product name does not include the VRAM. Every other RTX 5080 in the data has 16 GB, so we set it to 16 manually.
- **PNY NVIDIA RTX PRO SYNC** (id 4500148): this is a display sync accessory, not a graphics card, so it has no VRAM and stays empty.

In [27]:
is_id_4761812 = df["id"] == 4761812
df.loc[is_id_4761812, "vram"] = 16

In [28]:
df[df["vram"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram
103,4500148,PNY NVIDIA RTX PRO SYNC,999.0,2026-10-22,PNY,NVIDIA,True,NaN


In [29]:
df.head(20)

,id,name,price,date,brand,gpu_maker,is_pro,vram
0,4753925,MSI GeForce RTX 3050 VENTUS 2X E OC 6 GB,269.0,2026-10-22,MSI,NVIDIA,False,6.0
1,4796691,PNY Nvidia RTX PRO 6000 Blackwell 96 GB,19999.0,2026-10-22,PNY,NVIDIA,True,96.0
2,5339936,Asus GeForce RTX 5090 ROG Astral Edition 20 32 GB,7999.0,2026-10-22,Asus,NVIDIA,False,32.0
3,4791908,MSI GeForce RTX RTX 5060 SHADOW 2X OC 8 GB,429.0,2026-10-22,MSI,NVIDIA,False,8.0
4,4810066,Sapphire Radeon RX 9060 XT NITRO+ OC 16 GB,759.0,2026-10-22,Sapphire,AMD,False,16.0
5,4810090,Sapphire Radeon RX 9060 XT PULSE OC 16 GB,719.0,2026-10-22,Sapphire,AMD,False,16.0
6,4699440,Sapphire Radeon RX 9070 XT NITRO+ OC 16 GB,1199.0,2026-10-22,Sapphire,AMD,False,16.0
7,4820118,Gigabyte GeForce GT 710D3 2 GB,69.9,2026-10-22,Gigabyte,NVIDIA,False,2.0
8,5337283,Asus GeForce RTX 5060 Dual OC Advanced 8 GB,599.0,2026-10-22,Asus,NVIDIA,False,8.0
9,5164052,Asus Radeon RX 9070 GRE PRIME OC 12 GB,749.0,2026-10-22,Asus,AMD,False,12.0


In [30]:
df["name"].tolist()

['MSI GeForce RTX 3050 VENTUS 2X E OC 6 GB',
 'PNY Nvidia RTX PRO 6000 Blackwell 96 GB',
 'Asus GeForce RTX 5090 ROG Astral Edition 20 32 GB',
 'MSI GeForce RTX RTX 5060 SHADOW 2X OC 8 GB',
 'Sapphire Radeon RX 9060 XT NITRO+ OC 16 GB',
 'Sapphire Radeon RX 9060 XT PULSE OC 16 GB',
 'Sapphire Radeon RX 9070 XT NITRO+ OC 16 GB',
 'Gigabyte GeForce GT 710D3 2 GB',
 'Asus GeForce RTX 5060 Dual OC Advanced 8 GB',
 'Asus Radeon RX 9070 GRE PRIME OC 12 GB',
 'Gigabyte GeForce RTX 5070 TI GAMING OC 16 GB',
 'MSI GeForce RTX 5070 Ti GAMING TRIO OC 16 GB',
 'MSI GeForce RTX 5080 Ventus 3X White OC',
 'Sapphire Radeon RX 9070 XT PULSE 16 GB',
 'Asrock Intel ARC A380 Challenger ITX OC 6 GB',
 'Gigabyte GeForce GT 1030 Low Profile D4 2 GB',
 'Gigabyte GeForce RTX 5070 GAMING OC 12 GB',
 'Gigabyte GeForce RTX RTX 5060 Windforce OC 8 GB',
 'Asus GeForce RTX 5070 TI TUF GAMING OC 16 GB',
 'Gigabyte GeForce RTX 3060 Windforce OC 12 GB',
 'Gigabyte GeForce RTX RTX 5060 Windforce Max OC 8 GB',
 'Gigabyt

In [31]:
df["model"] = df["name"].str.extract(r"(?i)((?:RTX|GTX|GT)\s+\d+(?:\sTi)?)", expand=False).str.upper()

In [32]:
df["model"].value_counts()

model
RTX 5070 TI    10
RTX 5070       10
RTX 5060        8
RTX 5080        8
RTX 5060 TI     8
RTX 3050        7
RTX 5050        5
RTX 5090        4
RTX 3060        3
RTX 4000        2
GT 710          1
GT 1030         1
RTX  5070       1
RTX 2000        1
RTX 6000        1
RTX 4500        1
RTX 5000        1
RTX 3080        1
RTX 4060        1
RTX 4060 TI     1
Name: count, dtype: int64

In [33]:
df[(df["gpu_maker"].str.upper() == "NVIDIA")&(df["model"].isna())]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model
1,4796691,PNY Nvidia RTX PRO 6000 Blackwell 96 GB,19999.0,2026-10-22,PNY,NVIDIA,True,96.0,NaN
50,4796683,PNY Nvidia RTX PRO 6000 Blackwell Max-Q 96 GB,19999.0,2026-10-22,PNY,NVIDIA,True,96.0,NaN
64,4590023,PNY Quadro A400 Low Profile 4 GB,349.0,2026-10-22,PNY,NVIDIA,True,4.0,NaN
70,4894960,PNY Nvidia RTX PRO 5000 Blackwell 48 GB,10999.0,2026-10-22,PNY,NVIDIA,True,48.0,NaN
84,4894928,PNY Nvidia RTX PRO 2000 Blackwell 16 GB,2299.0,2026-10-22,PNY,NVIDIA,True,16.0,NaN
91,4894936,PNY Nvidia RTX PRO 4000 SFF Blackwell 24 GB,4199.0,2026-10-22,PNY,NVIDIA,True,24.0,NaN
92,4894952,PNY Nvidia RTX PRO 4500 Blackwell 32 GB,6499.0,2026-10-22,PNY,NVIDIA,True,32.0,NaN
99,4590031,PNY Quadro A1000 Low Profile 8 GB,799.0,2026-10-22,PNY,NVIDIA,True,8.0,NaN
100,4894944,PNY Nvidia RTX PRO 4000 Blackwell 24 GB,4199.0,2026-10-22,PNY,NVIDIA,True,24.0,NaN
103,4500148,PNY NVIDIA RTX PRO SYNC,999.0,2026-10-22,PNY,NVIDIA,True,NaN,NaN


## GPU model
The `model` column holds the GPU chip, for example `RTX 5070 TI` or `GT 1030`. It is extracted from the product name with a regex and converted to upper case, so that `Ti` and `TI` count as the same model.
**NVIDIA:** the pattern is `RTX`, `GTX` or `GT`, then one or more spaces, then the number, and optionally `Ti`. One or more spaces are allowed because some names have a double space (e.g. `RTX  5070`).
**Professional cards are not covered on purpose.** Their names follow other patterns (`RTX PRO 6000`, `Quadro A400`), and they are outside the scope of this project, which focuses on gaming cards. Most of them have no `model`, and the few that got one (e.g. `RTX 4000`) should not be trusted. Use `is_pro` to exclude them.

In [34]:
df[df["gpu_maker"] == "AMD"]["name"].tolist()

['Sapphire Radeon RX 9060 XT NITRO+ OC 16 GB',
 'Sapphire Radeon RX 9060 XT PULSE OC 16 GB',
 'Sapphire Radeon RX 9070 XT NITRO+ OC 16 GB',
 'Asus Radeon RX 9070 GRE PRIME OC 12 GB',
 'Sapphire Radeon RX 9070 XT PULSE 16 GB',
 'Sapphire Radeon RX 9070 PURE OC 16 GB',
 'Sapphire Radeon RX 7600 PULSE OC 8 GB',
 'Sapphire Radeon RX 9070 XT PURE OC 16 GB',
 'Sapphire Radeon RX 9070 GRE Gaming OC 12 GB',
 'Asrock Radeon RX 9060XT Challenger OC 16 GB',
 'Gigabyte Radeon RX 9070 XT GAMING OC 16 GB',
 'Asrock Radeon RX 7600 Challenger OC 8 GB',
 'Asus Radeon RX 9060 XT Dual OC 8 GB',
 'Asus Radeon RX 9060 XT Prime OC 8 GB',
 'Sapphire Radeon RX 9060 XT PURE OC 16 GB',
 'Gigabyte Radeon RX 9070 XT AORUS ELITE 16 GB',
 'Gigabyte Radeon RX 9060 XT GAMING OC 8 GB',
 'Gigabyte Radeon AI PRO R9700 AI TOP 32 GB',
 'Gigabyte Radeon RX 9060 XT Gaming OC 16 GB',
 'Asus Radeon AI PRO R9700 Turbo 32 GB',
 'AMD Radeon PRO W7500 8 GB',
 'AMD Radeon PRO W7700 16 GB',
 'XFX Radeon AI PRO R9700 32 GB',
 'Sapph

In [35]:
model_amd = df["name"].str.extract(r"(?i)(RX\s+\d+(?:\s?(?:XT|GRE))?)", expand=False).str.upper()
df["model"] = df["model"].fillna(model_amd)

In [36]:
df["model"].value_counts()

model
RTX 5070 TI    10
RTX 5070       10
RTX 5060        8
RX 9060 XT      8
RTX 5080        8
RTX 5060 TI     8
RTX 3050        7
RX 9070 XT      5
RTX 5050        5
RTX 5090        4
RTX 3060        3
RX 9070         3
RX 9070 GRE     2
RX 7600         2
RTX 4000        2
GT 710          1
GT 1030         1
RX 9060XT       1
RTX  5070       1
RTX 2000        1
RTX 6000        1
RTX 4500        1
RTX 5000        1
RX 6900 XT      1
RX 6500 XT      1
RTX 3080        1
RTX 4060        1
RTX 4060 TI     1
Name: count, dtype: int64

In [37]:
df["model"].value_counts()

model
RTX 5070 TI    10
RTX 5070       10
RTX 5060        8
RX 9060 XT      8
RTX 5080        8
RTX 5060 TI     8
RTX 3050        7
RX 9070 XT      5
RTX 5050        5
RTX 5090        4
RTX 3060        3
RX 9070         3
RX 9070 GRE     2
RX 7600         2
RTX 4000        2
GT 710          1
GT 1030         1
RX 9060XT       1
RTX  5070       1
RTX 2000        1
RTX 6000        1
RTX 4500        1
RTX 5000        1
RX 6900 XT      1
RX 6500 XT      1
RTX 3080        1
RTX 4060        1
RTX 4060 TI     1
Name: count, dtype: int64

In [38]:
df[df["model"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model
1,4796691,PNY Nvidia RTX PRO 6000 Blackwell 96 GB,19999.0,2026-10-22,PNY,NVIDIA,True,96.0,NaN
14,4062272,Asrock Intel ARC A380 Challenger ITX OC 6 GB,179.0,2026-10-22,Asrock,INTEL,False,6.0,NaN
48,4699505,Intel Intel ARC A60 Graphics 12 GB,399.0,2026-10-22,Intel,INTEL,False,12.0,NaN
50,4796683,PNY Nvidia RTX PRO 6000 Blackwell Max-Q 96 GB,19999.0,2026-10-22,PNY,NVIDIA,True,96.0,NaN
64,4590023,PNY Quadro A400 Low Profile 4 GB,349.0,2026-10-22,PNY,NVIDIA,True,4.0,NaN
67,4998200,Gigabyte Radeon AI PRO R9700 AI TOP 32 GB,2699.0,2026-10-22,Gigabyte,AMD,True,32.0,NaN
70,4894960,PNY Nvidia RTX PRO 5000 Blackwell 48 GB,10999.0,2026-10-22,PNY,NVIDIA,True,48.0,NaN
78,4998219,Asus Radeon AI PRO R9700 Turbo 32 GB,2699.0,2026-10-22,Asus,AMD,True,32.0,NaN
79,4309510,AMD Radeon PRO W7500 8 GB,549.0,2026-10-22,AMD,AMD,True,8.0,NaN
84,4894928,PNY Nvidia RTX PRO 2000 Blackwell 16 GB,2299.0,2026-10-22,PNY,NVIDIA,True,16.0,NaN


In [39]:
df[df["gpu_maker"].str.upper() == "INTEL"]["name"].tolist()

['Asrock Intel ARC A380 Challenger ITX OC 6 GB',
 'Intel Intel ARC A60 Graphics 12 GB',
 'Asrock Intel ARC A310 Low Profile 4 GB']

In [40]:
df["gpu_maker"].value_counts()

gpu_maker
NVIDIA    88
AMD       35
INTEL      3
Name: count, dtype: int64

## Fixing a professional card that is_pro missed

`is_pro` looks for the word "Pro" in the card name. One card slips through:

- **Intel Intel ARC A60 Graphics 12 GB** (id 4699505): Plaisio leaves "Pro" out of the name, but the card's full name is Intel Arc Pro A60 12GB, a professional workstation card .

We set `is_pro` to True for this id by hand. We fix our own column and leave `name` as Plaisio wrote it, so the raw data stays unchanged.

The cells below show the row before the fix (False), the fix, and the row after it (True).

In [41]:
df[df["name"] == "Intel Intel ARC A60 Graphics 12 GB"]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model
48,4699505,Intel Intel ARC A60 Graphics 12 GB,399.0,2026-10-22,Intel,INTEL,False,12.0,NaN


In [42]:
is_id_4699505 = df["id"] == 4699505
df.loc[is_id_4699505, "is_pro"] = True

In [43]:
df[df["name"] == "Intel Intel ARC A60 Graphics 12 GB"]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model
48,4699505,Intel Intel ARC A60 Graphics 12 GB,399.0,2026-10-22,Intel,INTEL,True,12.0,NaN


In [44]:
model_intel = df["name"].str.extract(r"(?i)(ARC\s+[A-Z]\d+)", expand=False).str.upper()
df["model"] = df["model"].fillna(model_intel)

In [45]:
df["model"].value_counts()

model
RTX 5070 TI    10
RTX 5070       10
RTX 5060        8
RX 9060 XT      8
RTX 5080        8
RTX 5060 TI     8
RTX 3050        7
RX 9070 XT      5
RTX 5050        5
RTX 5090        4
RTX 3060        3
RX 9070         3
RX 9070 GRE     2
RX 7600         2
RTX 4000        2
GT 710          1
ARC A380        1
GT 1030         1
RX 9060XT       1
RTX  5070       1
ARC A60         1
RTX 2000        1
RTX 6000        1
RTX 4500        1
RTX 5000        1
ARC A310        1
RX 6900 XT      1
RX 6500 XT      1
RTX 3080        1
RTX 4060        1
RTX 4060 TI     1
Name: count, dtype: int64

In [46]:
df["model"] = df["model"].str.replace(" ", "")

In [47]:
df["model"].value_counts()

model
RTX5070      11
RTX5070TI    10
RX9060XT      9
RTX5060       8
RTX5080       8
RTX5060TI     8
RTX3050       7
RX9070XT      5
RTX5050       5
RTX5090       4
RTX3060       3
RX9070        3
RX9070GRE     2
RX7600        2
RTX4000       2
GT710         1
ARCA380       1
GT1030        1
ARCA60        1
RTX2000       1
RTX6000       1
RTX4500       1
RTX5000       1
ARCA310       1
RX6900XT      1
RX6500XT      1
RTX3080       1
RTX4060       1
RTX4060TI     1
Name: count, dtype: int64

## Check: professional cards marked as is_pro False

Professional cards are expected to have no model. This filter shows cards with no model that are **not** marked as pro, so it should return an empty table.

The first run found 5 Radeon AI PRO R9700 cards with `is_pro` False. The `is_pro` rule was then fixed to also match "Radeon AI PRO" (see above), and the table is now empty.

In [48]:
df[(df["model"].isna()) & (~df["is_pro"])]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model


In [49]:
df[df["model"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model
1,4796691,PNY Nvidia RTX PRO 6000 Blackwell 96 GB,19999.0,2026-10-22,PNY,NVIDIA,True,96.0,NaN
50,4796683,PNY Nvidia RTX PRO 6000 Blackwell Max-Q 96 GB,19999.0,2026-10-22,PNY,NVIDIA,True,96.0,NaN
64,4590023,PNY Quadro A400 Low Profile 4 GB,349.0,2026-10-22,PNY,NVIDIA,True,4.0,NaN
67,4998200,Gigabyte Radeon AI PRO R9700 AI TOP 32 GB,2699.0,2026-10-22,Gigabyte,AMD,True,32.0,NaN
70,4894960,PNY Nvidia RTX PRO 5000 Blackwell 48 GB,10999.0,2026-10-22,PNY,NVIDIA,True,48.0,NaN
78,4998219,Asus Radeon AI PRO R9700 Turbo 32 GB,2699.0,2026-10-22,Asus,AMD,True,32.0,NaN
79,4309510,AMD Radeon PRO W7500 8 GB,549.0,2026-10-22,AMD,AMD,True,8.0,NaN
84,4894928,PNY Nvidia RTX PRO 2000 Blackwell 16 GB,2299.0,2026-10-22,PNY,NVIDIA,True,16.0,NaN
85,4374797,AMD Radeon PRO W7700 16 GB,1249.0,2026-10-22,AMD,AMD,True,16.0,NaN
86,4998308,XFX Radeon AI PRO R9700 32 GB,2699.0,2026-10-22,XFX,AMD,True,32.0,NaN


In [50]:
def clean_data(df):
    df = df.copy()
    df["name"] = df["name"].str.replace("VGA ", "")
    df["brand"] = df["name"].str.split().str[0]
    
    amd = df["name"].str.contains("Radeon|AMD", case = False)
    nvidia = df["name"].str.contains("GeForce|Nvidia|PNY", case = False)
    intel = df["name"].str.contains("Intel|Arc", case = False)
    
    df.loc[amd, "gpu_maker"] = "AMD"
    df.loc[nvidia, "gpu_maker"] = "NVIDIA"
    df.loc[intel, "gpu_maker"] = "INTEL"
    
    amd_pro_cards = df["name"].str.contains("Radeon pro|Radeon AI PRO", case = False)
    nvidia_pro_cards = df["name"].str.contains("Quadro|RTX PRO|", case = False)
    intel_pro_cards = df["name"].str.contains("Arc Pro", case = False)
    df["is_pro"] = (amd_pro_cards|nvidia_pro_cards|intel_pro_cards)

    is_id_4699505 = df["id"] == 4699505
    df.loc[is_id_4699505, "is_pro"] = True

    df["vram"] = df["name"].str.extract(r"(\d+)\s?GB?\b").astype(float)

    is_id_4761812 = df["id"] == 4761812
    df.loc[is_id_4761812, "vram"] = 16

    df["model"] = df["name"].str.extract(r"(?i)((?:RTX|GTX|GT)\s+\d+(?:\sTi)?)", expand=False).str.upper()
    
    model_amd = df["name"].str.extract(r"(?i)(RX\s+\d+(?:\s?(?:XT|GRE))?)", expand=False).str.upper()
    df["model"] = df["model"].fillna(model_amd)
    model_intel = df["name"].str.extract(r"(?i)(ARC\s+[A-Z]\d+)", expand=False).str.upper()
    df["model"] = df["model"].fillna(model_intel)
    df["model"] = df["model"].str.replace(" ", "")
    return df

In [51]:
raw = pd.read_csv("../data/raw/2026-10-22.csv")

In [52]:
clean_data(raw)

,id,name,price,date,brand,gpu_maker,is_pro,vram,model
0,4753925,MSI GeForce RTX 3050 VENTUS 2X E OC 6 GB,269.0,2026-10-22,MSI,NVIDIA,True,6.0,RTX3050
1,4796691,PNY Nvidia RTX PRO 6000 Blackwell 96 GB,19999.0,2026-10-22,PNY,NVIDIA,True,96.0,NaN
2,5339936,Asus GeForce RTX 5090 ROG Astral Edition 20 32 GB,7999.0,2026-10-22,Asus,NVIDIA,True,32.0,RTX5090
3,4791908,MSI GeForce RTX RTX 5060 SHADOW 2X OC 8 GB,429.0,2026-10-22,MSI,NVIDIA,True,8.0,RTX5060
4,4810066,Sapphire Radeon RX 9060 XT NITRO+ OC 16 GB,759.0,2026-10-22,Sapphire,AMD,True,16.0,RX9060XT
...,...,...,...,...,...,...,...,...,...
121,4699491,Sapphire Radeon RX 9070 PULSE 16 GB,899.0,2026-10-22,Sapphire,AMD,True,16.0,RX9070
122,4604164,Asus GeForce RTX 4060 Ti DUAL EVO OC 8 GB,449.0,2026-10-22,Asus,NVIDIA,True,8.0,RTX4060TI
123,4781376,Asus GeForce RTX 5070 ROG STRIX OC 12 GB,999.0,2026-10-22,Asus,NVIDIA,True,12.0,RTX5070
124,4811798,Asus Radeon RX 9060 XT Prime OC 16 GB,649.0,2026-10-22,Asus,AMD,True,16.0,RX9060XT


In [53]:
df[(df["model"].isna()) & (~df["is_pro"])]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model


In [54]:
df["model"].value_counts()

model
RTX5070      11
RTX5070TI    10
RX9060XT      9
RTX5060       8
RTX5080       8
RTX5060TI     8
RTX3050       7
RX9070XT      5
RTX5050       5
RTX5090       4
RTX3060       3
RX9070        3
RX9070GRE     2
RX7600        2
RTX4000       2
GT710         1
ARCA380       1
GT1030        1
ARCA60        1
RTX2000       1
RTX6000       1
RTX4500       1
RTX5000       1
ARCA310       1
RX6900XT      1
RX6500XT      1
RTX3080       1
RTX4060       1
RTX4060TI     1
Name: count, dtype: int64

In [55]:
df["gpu_maker"].value_counts()

gpu_maker
NVIDIA    88
AMD       35
INTEL      3
Name: count, dtype: int64

In [56]:
df[df["vram"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model
103,4500148,PNY NVIDIA RTX PRO SYNC,999.0,2026-10-22,PNY,NVIDIA,True,NaN,NaN


In [57]:
cleaned = clean_data(raw)

In [58]:
cleaned[(cleaned["model"].isna()) & (~cleaned["is_pro"])]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model


In [59]:
cleaned["model"].value_counts()

model
RTX5070      11
RTX5070TI    10
RX9060XT      9
RTX5060       8
RTX5080       8
RTX5060TI     8
RTX3050       7
RX9070XT      5
RTX5050       5
RTX5090       4
RTX3060       3
RX9070        3
RX9070GRE     2
RX7600        2
RTX4000       2
GT710         1
ARCA380       1
GT1030        1
ARCA60        1
RTX2000       1
RTX6000       1
RTX4500       1
RTX5000       1
ARCA310       1
RX6900XT      1
RX6500XT      1
RTX3080       1
RTX4060       1
RTX4060TI     1
Name: count, dtype: int64

In [60]:
cleaned["gpu_maker"].value_counts()

gpu_maker
NVIDIA    88
AMD       35
INTEL      3
Name: count, dtype: int64

In [61]:
cleaned[cleaned["gpu_maker"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model


In [62]:
raw_2 = pd.read_csv("../data/raw/2026-10-25.csv")

In [63]:
cleaned = clean_data(raw_2)

In [ ]:
cleaned[(cleaned["model"].isna()) & (~cleaned["is_pro"])]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model


In [ ]:
cleaned[cleaned["gpu_maker"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model


In [ ]:
cleaned["model"].value_counts()

model
RX9060XT     9
RTX5070TI    9
RTX5070      9
RTX5080      8
RTX3050      7
RTX5060      6
RTX5060TI    6
RX9070XT     5
RTX5050      5
RTX5090      4
RX9070       3
RTX3060      3
RX9070GRE    2
RX7600       2
RTX4000      2
GT710        1
ARCA380      1
GT1030       1
ARCA60       1
RTX2000      1
RTX6000      1
RTX4500      1
RTX5000      1
ARCA310      1
RX6900XT     1
RX6500XT     1
RTX3080      1
RTX4060      1
RTX4060TI    1
Name: count, dtype: int64

In [ ]:
cleaned["gpu_maker"].value_counts()

gpu_maker
NVIDIA    81
AMD       35
INTEL      3
Name: count, dtype: int64

## Combining all days
Each day's csv is all_days  with `clean_data` and all days are stacked into one table, so every card has one row per day. The files are found automatically, so a new day's csv is picked up without changing the code.

In [68]:
import glob

raw_files = sorted(glob.glob("../data/raw/*.csv"))
print(raw_files)

['../data/raw\\2026-10-21.csv', '../data/raw\\2026-10-22.csv', '../data/raw\\2026-10-23.csv', '../data/raw\\2026-10-25.csv', '../data/raw\\2026-10-26.csv', '../data/raw\\2026-10-27.csv', '../data/raw\\2026-10-28.csv', '../data/raw\\2026-10-29.csv', '../data/raw\\2026-10-30.csv']


In [69]:
cleaned_days = []
for file in raw_files:
    raw = pd.read_csv(file)
    cleaned = clean_data(raw)
    cleaned_days.append(cleaned)
print(len(cleaned_days))

9


In [70]:
all_days = pd.concat(cleaned_days, ignore_index=True)


In [74]:
all_days["date"].value_counts().sort_index()

date
2026-10-21    123
2026-10-22    126
2026-10-23    126
2026-10-25    119
2026-10-26    118
2026-10-27    116
2026-10-28    119
2026-10-29    119
2026-10-30    114
Name: count, dtype: int64

In [75]:
all_days[(all_days["model"].isna()) & (~all_days["is_pro"])]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model


In [77]:
all_days[all_days["gpu_maker"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram,model
